# 1

In [2]:
import pandas as pd

file_path = "Данные для тестового задания.xlsx"
df = pd.read_excel(file_path, sheet_name="Данные об аудитории")

df["date"] = pd.to_datetime(df["date"])

# оставляем только ноябрь 2023
november_df = df[(df["date"].dt.month == 11) & (df["date"].dt.year == 2023)]

mau = november_df["user_id"].nunique()

print("MAU =", mau)

MAU = 7639


# 2

In [3]:
# считаем DAU по дням (уникальные пользователи в день)
dau_per_day = df.groupby("date")["user_id"].nunique()

# средний DAU за месяц
avg_dau = dau_per_day.mean()

print("DAU (средний) =", round(avg_dau, 2))

DAU (средний) = 560.47


# 3

In [4]:
# 1. находим первый визит пользователя (дата регистрации)
first_visit = df.groupby("user_id")["date"].min().reset_index()
first_visit.columns = ["user_id", "first_date"]

# 2. берем пользователей, пришедших 1 ноября
cohort = first_visit[first_visit["first_date"] == "2023-11-01"]["user_id"]

# 3. проверяем, кто из них вернулся 2 ноября
returned = df[
    (df["user_id"].isin(cohort)) &
    (df["date"] == "2023-11-02")
]["user_id"].nunique()

# 4. считаем retention
retention = returned / cohort.nunique() * 100

print("Retention Day 1 =", round(retention, 1), "%")

Retention Day 1 = 26.6 %


# 5

In [5]:
# считаем всех пользователей
total_users = df["user_id"].nunique()

# пользователи, у которых был хотя бы 1 просмотр
users_with_views = df[df["view_adverts"] > 0]["user_id"].nunique()

# конверсия
conversion = users_with_views / total_users * 100

print("Conversion =", round(conversion, 1), "%")

Conversion = 46.3 %


# 6

In [6]:
# считаем суммарные просмотры на пользователя
views_per_user = df.groupby("user_id")["view_adverts"].sum()

# среднее
avg_views = views_per_user.mean()

print("Среднее просмотров =", round(avg_views, 1))

Среднее просмотров = 2.9


# 7

In [7]:
promoters = 1200 / 2000 * 100  # 60%
detractors = 500 / 2000 * 100  # 25%

nps = promoters - detractors
print(nps)

35.0


# 8

In [11]:
import pandas as pd
from scipy.stats import ttest_ind

file_path = "Данные для тестового задания.xlsx"
df = pd.read_excel(file_path, sheet_name="Данные АБ тестов")

df.columns = df.columns.str.strip()
df["experiment_group"] = df["experiment_group"].astype(str).str.strip()

results = {}

for exp in sorted(df["experiment_num"].unique()):
    exp_df = df[df["experiment_num"] == exp]
    
    groups = exp_df["experiment_group"].unique()
    print(f"Эксперимент {exp}, группы: {groups}")
    
    group1 = exp_df[exp_df["experiment_group"] == groups[0]]["revenue"]
    group2 = exp_df[exp_df["experiment_group"] == groups[1]]["revenue"]
    
    stat, p_value = ttest_ind(group1, group2, equal_var=False)
    results[exp] = round(p_value, 4)

print(results)

Эксперимент 1, группы: ['test' 'control']
Эксперимент 2, группы: ['control' 'test']
Эксперимент 3, группы: ['control' 'test']
{np.int64(1): np.float64(0.689), np.int64(2): np.float64(0.0011), np.int64(3): np.float64(0.0603)}


In [13]:
# Эксперимент 1:
# p-value = 0.689. Статистически значимых различий между группами нет, так как p-value > 0.05. Изменение не дало эффекта.

# Эксперимент 2:
# p-value = 0.0011. Статистически значимые различия есть, так как p-value < 0.05. Изменение можно считать успешным и рекомендовать к внедрению.

# Эксперимент 3:
# p-value = 0.0603. Статистически значимых различий на уровне значимости 0.05 нет, хотя результат близок к порогу. Однозначный вывод делать рано, желательно продолжить тест или увеличить выборку.

# Итоговые рекомендации:
# Рекомендуется внедрить только эксперимент 2.
# Эксперимент 1 — не внедрять.
# Эксперимент 3 — повторно проверить на большей выборке или продлить тест.

# 9

In [14]:
import pandas as pd

# загрузка
file_path = "Данные для тестового задания.xlsx"
df = pd.read_excel(file_path, sheet_name="Листеры")

# если у пользователя несколько записей — агрегируем
revenue_per_user = df.groupby("user_id")["revenue"].sum()

# средний доход
avg_revenue = revenue_per_user.mean()

print(round(avg_revenue, 1))

156.5


# 10

In [15]:
import pandas as pd

# загрузка
file_path = "Данные для тестового задания.xlsx"
df = pd.read_excel(file_path, sheet_name="Листеры")

# если есть дубликаты пользователей — берём по одному значению возраста
age_per_user = df.groupby("user_id")["age"].first()

# медиана
median_age = age_per_user.median()

print(median_age)

28.0


# 18

In [20]:
from statsmodels.stats.proportion import proportions_ztest

# платежи
payments_A = 1003
payments_B = 1099

# пользователи
users_A = 100047501
users_B = 100001055

# считаем p-value
stat, p_value = proportions_ztest([payments_A, payments_B],[users_A, users_B])

print(p_value)

0.03533044544854254


In [21]:
if p_value < 0.05:
    print("Разница есть (значимо)")
else:
    print("Разницы нет")

Разница есть (значимо)


In [ ]:
# Я сравнила конверсии между контрольной и тестовой группой с помощью z-теста для долей.
# Получила p-value и проверила его относительно уровня значимости 0.05.
# Если p-value меньше 0.05, разница считается статистически значимой.